# Entry-quality gates: corrected before/after

## tl;dr

- **Production headline:** 474 persisted evaluation rows contain 2 unique `trade_ready` decisions. Strict replay produces **1 simulated fill, +$800**; the put/down decision is **`target_before_entry`**, so it has no entry and no PnL. With only one replayed trade, this is not evidence of an edge.
- **Corrected S2 observational proxy:** 233 unique candidates → **7 replay rows (3.00%)**, 5 wins, **+$290**. Down/put is 3 rows, 2 wins, +$120. S2 is explicitly excluded from production-strategy PnL.
- **GTH lifecycle bug remains directly evidenced:** 2 of 6 historical GTH virtual opens occur after the reconstructed 600-second validity limit (826.461s and 5,261.461s).
- **Legacy results are retired:** 267/12/-$1,560 S2, GTH 12/-$3,250, and put/down 8/-$1,730 came from the superseded diagnostic and must not be used as current strategy evidence.

`automatic_ordering=False` remains unchanged. All dollar results here are quote-lake replays, not live executions.

## Context & Methods

The primary source is the corrected output:

`/srv/data/spx-spark/data/reports/odte_level_backtest/validation=2026-07-18-entry-quality/cutoff=2026-07-17/`

The prior `validation=2026-07-18` output is loaded only for a clearly labeled before/after audit. Feature-store inputs are `trade_intents`, `gth_dip_reclaim`, and `virtual_strategy`.

### Key assumptions

1. The fixed window ends at **2026-07-18 00:00:00 UTC exclusive**; later partitions and timestamps are excluded.
2. Only `trade_ready × baseline × naked` is the production-strategy cohort. Confirmed, prefill/S2, and GTH-dip sets are controls or observational proxies.
3. Strict `trade_ready` replay uses the persisted provider, contract, limit, exclusive expiry window, and fails closed if target/invalidation occurs before fill.
4. Corrected S2 deduplicates repeated production semantic keys to the earliest valid first touch. It is follow-through-only and cannot substitute for `trade_ready`.
5. RTH is 09:30–16:00 America/New_York, derived from timezone-aware timestamps.
6. `h60_ret`, `h300_ret`, `h900_ret`, and virtual post-entry horizon outcomes are never used to define a gate or cohort.
7. A win is `pnl_usd > 0`; flat is not a win.

In [1]:
from collections import Counter, defaultdict
from csv import DictReader
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from statistics import median
from zoneinfo import ZoneInfo
import json

DATA_ROOT = Path("/srv/data/spx-spark/data")
REPORT_ROOT = DATA_ROOT / "reports/odte_level_backtest"
CURRENT_DIR = REPORT_ROOT / "validation=2026-07-18-entry-quality/cutoff=2026-07-17"
LEGACY_DIR = REPORT_ROOT / "validation=2026-07-18/cutoff=2026-07-17"
FEATURES_ROOT = DATA_ROOT / "features"
TRADE_INTENTS_ROOT = FEATURES_ROOT / "trade_intents"
GTH_SIGNALS_ROOT = FEATURES_ROOT / "gth_dip_reclaim"
VIRTUAL_ROOT = FEATURES_ROOT / "virtual_strategy"

CUTOFF_DATE = date(2026, 7, 17)
CUTOFF_AT = datetime(2026, 7, 18, tzinfo=timezone.utc)
NEW_YORK = ZoneInfo("America/New_York")
GTH_TTL_SECONDS = 600.0
CLOCK_TOLERANCE_SECONDS = 5.0

for required in (
    CURRENT_DIR / "artifact.json", CURRENT_DIR / "trades.csv",
    LEGACY_DIR / "artifact.json", LEGACY_DIR / "trades.csv",
    TRADE_INTENTS_ROOT, GTH_SIGNALS_ROOT, VIRTUAL_ROOT,
):
    assert required.exists(), f"missing source: {required}"

print(f"Primary backtest: {CURRENT_DIR}")
print(f"Legacy comparison only: {LEGACY_DIR}")
print(f"Cutoff (exclusive): {CUTOFF_AT.isoformat()}")
print("Dependencies: Python standard library only")


Primary backtest: /srv/data/spx-spark/data/reports/odte_level_backtest/validation=2026-07-18-entry-quality/cutoff=2026-07-17
Legacy comparison only: /srv/data/spx-spark/data/reports/odte_level_backtest/validation=2026-07-18/cutoff=2026-07-17
Cutoff (exclusive): 2026-07-18T00:00:00+00:00
Dependencies: Python standard library only


## Data

The following loader applies partition and record-time cutoffs and records the exact source coverage.

In [2]:
def parse_ts(value):
    if not value:
        return None
    parsed = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError(f"naive timestamp: {value}")
    return parsed.astimezone(timezone.utc)

def partition_date(path):
    return date.fromisoformat(path.parent.name.split("=", 1)[1])

def load_jsonl(root, time_fields):
    rows, paths = [], []
    for path in sorted(root.glob("date=*/events.jsonl")):
        if partition_date(path) > CUTOFF_DATE:
            continue
        paths.append(path)
        with path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                row = json.loads(line)
                primary = next((parse_ts(row.get(field)) for field in time_fields if row.get(field)), None)
                if primary is not None and primary >= CUTOFF_AT:
                    continue
                row["_primary_at"] = primary
                row["_source_path"] = str(path)
                row["_line_number"] = line_number
                rows.append(row)
    return rows, paths

def load_csv(path):
    with path.open(newline="", encoding="utf-8") as handle:
        return [row for row in DictReader(handle) if parse_ts(row["at"]) < CUTOFF_AT]

def load_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

def md_table(headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    widths = [len(str(value)) for value in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(value))
    def render(values):
        return "| " + " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(values)) + " |"
    print(render(headers))
    print("| " + " | ".join("-" * width for width in widths) + " |")
    for row in rows:
        print(render(row))

def canonical_bucket(value):
    local = parse_ts(value).astimezone(NEW_YORK)
    minute = local.hour * 60 + local.minute
    if 570 <= minute < 630:
        return "rth_open"
    if 630 <= minute < 900:
        return "rth_midday"
    if 900 <= minute < 960:
        return "rth_close"
    return "gth"

def market_session(value):
    return "RTH" if canonical_bucket(value).startswith("rth_") else "GTH"

def summarize(rows):
    count = len(rows)
    wins = sum(float(row["pnl_usd"]) > 0 for row in rows)
    total = sum(float(row["pnl_usd"]) for row in rows)
    return count, wins, (wins / count if count else 0.0), total, (total / count if count else 0.0)

current_artifact = load_json(CURRENT_DIR / "artifact.json")
legacy_artifact = load_json(LEGACY_DIR / "artifact.json")
current_trades = load_csv(CURRENT_DIR / "trades.csv")
legacy_trades = load_csv(LEGACY_DIR / "trades.csv")
trade_intents, intent_paths = load_jsonl(TRADE_INTENTS_ROOT, ("evaluated_at",))
gth_signals, gth_paths = load_jsonl(GTH_SIGNALS_ROOT, ("confirmed_at",))
virtual_events, virtual_paths = load_jsonl(VIRTUAL_ROOT, ("opened_at", "observed_at", "closed_at", "at"))

assert parse_ts(current_artifact["window"]["cutoff_at"]) == CUTOFF_AT
assert current_artifact["method"]["production_strategy_set"] == "trade_ready"
md_table(
    ("source", "rows", "role"),
    (
        ("corrected trades.csv", len(current_trades), "primary replay ledger"),
        ("legacy trades.csv", len(legacy_trades), "before-only invalid diagnostic"),
        ("trade_intents", len(trade_intents), "production decision telemetry"),
        ("gth_dip_reclaim", len(gth_signals), "GTH confirmations"),
        ("virtual_strategy", len(virtual_events), "lifecycle events"),
    ),
)
print(f"Corrected artifact generated_at: {current_artifact['generated_at']}")
print(f"Complete sessions: {', '.join(current_artifact['window']['complete_sessions'])}")


| source               | rows | role                           |
| -------------------- | ---- | ------------------------------ |
| corrected trades.csv | 307  | primary replay ledger          |
| legacy trades.csv    | 363  | before-only invalid diagnostic |
| trade_intents        | 474  | production decision telemetry  |
| gth_dip_reclaim      | 6    | GTH confirmations              |
| virtual_strategy     | 32   | lifecycle events               |
Corrected artifact generated_at: 2026-07-18T12:50:46.727912+00:00
Complete sessions: 2026-07-13, 2026-07-14, 2026-07-15, 2026-07-16, 2026-07-17


## Results

### Current production and S2 proxy

Production and S2 are deliberately reported separately. The first table is the decision/replay funnel; the second traces both production-ready decisions.

In [3]:
coverage = current_artifact["trade_intent_coverage"]
status_counts = Counter(row["status"] for row in trade_intents)
ready_rows = sorted(
    (row for row in trade_intents if row["status"] == "trade_ready"),
    key=lambda row: row["evaluated_at"],
)
production = current_artifact["production_strategy_total"]["result"]
production_rows = [
    row for row in current_trades
    if row["set_name"] == "trade_ready" and row["profile"] == "baseline" and row["variant"] == "naked"
]
production_skips = {
    row["key"]: row["reason"]
    for row in current_artifact["skipped_detail"]
    if row["set_name"] == "trade_ready" and row["profile"] == "baseline" and row["variant"] == "naked"
}
s2 = current_artifact["profiles"]["baseline"]["prefill"]["variants"]["naked"]
s2_rows = [
    row for row in current_trades
    if row["set_name"] == "prefill" and row["profile"] == "baseline" and row["variant"] == "naked"
]

assert coverage["records_by_status"] == dict(status_counts)
assert coverage["distinct_trade_ready_intent_ids"] == len(ready_rows) == 2
assert current_artifact["signal_counts"]["trade_ready"] == 2
assert len(production_rows) == production["n"] == 1
assert production["total_pnl_usd"] == 800.0
assert production["skipped"] == {"target_before_entry": 1}
assert current_artifact["signal_counts"]["prefill"] == 233
assert len(s2_rows) == s2["n"] == 7
assert s2["total_pnl_usd"] == 290.0
assert s2["skipped"] == {"follow_through_failed": 109, "follow_through_unavailable": 117}

md_table(
    ("cohort / state", "count", "rate", "replay PnL"),
    (
        ("production evaluation records", coverage["evaluation_records"], "descriptive rows", "not PnL cohort"),
        ("production observing", status_counts["observing"], "non-decision telemetry", "excluded"),
        ("production blocked", status_counts["blocked"], "repeated decisions", "excluded"),
        ("production trade_ready intents", 2, "2 unique", "strict replay eligible"),
        ("production strict fills", 1, "50.00% of ready", "USD +800"),
        ("production target_before_entry", 1, "50.00% of ready", "no entry / no PnL"),
        ("S2 unique observational candidates", 233, "deduped proxy", "not production"),
        ("S2 follow-through failed", 109, f"{109 / 233:.2%}", "no proxy entry"),
        ("S2 gate data unavailable", 117, f"{117 / 233:.2%}", "fail closed"),
        ("S2 observational replay rows", 7, f"{7 / 233:.2%}", "USD +290"),
    ),
)
print()
production_by_intent = {row["key"]: row for row in production_rows}
md_table(
    ("intent_id", "direction", "contract", "strict replay", "auto_order"),
    [
        (
            row["intent_id"],
            row["direction"],
            row["contract_id"],
            (
                f"fill {production_by_intent[row['intent_id']]['entry_px']} → "
                f"{production_by_intent[row['intent_id']]['exit_px']}; "
                f"USD {float(production_by_intent[row['intent_id']]['pnl_usd']):+.0f}"
                if row["intent_id"] in production_by_intent
                else production_skips.get(row["intent_id"], "unavailable")
            ),
            row["automatic_ordering"],
        )
        for row in ready_rows
    ],
)


| cohort / state                     | count | rate                   | replay PnL             |
| ---------------------------------- | ----- | ---------------------- | ---------------------- |
| production evaluation records      | 474   | descriptive rows       | not PnL cohort         |
| production observing               | 452   | non-decision telemetry | excluded               |
| production blocked                 | 20    | repeated decisions     | excluded               |
| production trade_ready intents     | 2     | 2 unique               | strict replay eligible |
| production strict fills            | 1     | 50.00% of ready        | USD +800               |
| production target_before_entry     | 1     | 50.00% of ready        | no entry / no PnL      |
| S2 unique observational candidates | 233   | deduped proxy          | not production         |
| S2 follow-through failed           | 109   | 46.78%                 | no proxy entry         |
| S2 gate data unavailable    

### Corrected slices and before/after

Current S2 slices remain observational. The legacy figures below are retained only to show why the earlier strategy diagnosis was withdrawn.

In [4]:
slice_rows = []
for cohort_name, rows in (("S2 observational", s2_rows), ("production strict fill", production_rows)):
    for market in ("GTH", "RTH"):
        for direction in ("down", "up"):
            selected = [row for row in rows if market_session(row["at"]) == market and row["direction"] == direction]
            if not selected:
                continue
            count, wins, win_rate, total, average = summarize(selected)
            slice_rows.append((
                cohort_name, market, direction, count, wins, f"{win_rate:.1%}",
                f"USD {total:+,.0f}", f"USD {average:+,.2f}",
            ))
md_table(
    ("cohort", "session", "direction", "rows", "wins", "win rate", "total", "average"),
    slice_rows,
)

legacy_s2 = legacy_artifact["profiles"]["baseline"]["prefill"]["variants"]["naked"]
legacy_s2_rows = [
    row for row in legacy_trades
    if row["set_name"] == "prefill" and row["profile"] == "baseline" and row["variant"] == "naked"
]
legacy_baseline = [
    row for row in legacy_trades
    if row["set_name"] in {"confirmed", "prefill", "gth_dip"}
    and row["profile"] == "baseline" and row["variant"] == "naked"
]
legacy_gth = [row for row in legacy_baseline if market_session(row["at"]) == "GTH"]
legacy_down = [row for row in legacy_s2_rows if row["direction"] == "down"]
current_down = [row for row in s2_rows if row["direction"] == "down"]
legacy_gth_stats = summarize(legacy_gth)
legacy_down_stats = summarize(legacy_down)
current_down_stats = summarize(current_down)

assert legacy_artifact["signal_counts"]["prefill"] == 267
assert legacy_s2["n"] == 12 and legacy_s2["total_pnl_usd"] == -1560.0
assert legacy_gth_stats[:2] == (12, 0) and legacy_gth_stats[3] == -3250.0
assert legacy_down_stats[:2] == (8, 1) and legacy_down_stats[3] == -1730.0
assert current_down_stats[:2] == (3, 2) and current_down_stats[3] == 120.0

print()
md_table(
    ("metric", "legacy superseded output", "corrected current output", "status"),
    (
        ("S2 candidates", "267 snapshots", "233 unique semantic keys", "current proxy only"),
        ("S2 failed / unavailable", "251 / 4", "109 / 117", "availability now fail closed"),
        ("S2 replay rows / PnL", "12 / USD -1,560", "7 / USD +290", "observational only"),
        ("S2 put/down", "8 rows; 1 win; USD -1,730", "3 rows; 2 wins; USD +120", "do not infer direction edge"),
        ("combined GTH diagnostic", "12 rows; 0 wins; USD -3,250", "retired mixed proxy total", "not production evidence"),
        ("production trade_ready", "not present", "2 intents; 1 fill; USD +800", "n=1, not edge"),
    ),
)
print("Legacy figures above are an error audit only; none are current headline metrics.")


| cohort                 | session | direction | rows | wins | win rate | total    | average     |
| ---------------------- | ------- | --------- | ---- | ---- | -------- | -------- | ----------- |
| S2 observational       | GTH     | down      | 1    | 1    | 100.0%   | USD +40  | USD +40.00  |
| S2 observational       | RTH     | down      | 2    | 1    | 50.0%    | USD +80  | USD +40.00  |
| S2 observational       | RTH     | up        | 4    | 3    | 75.0%    | USD +170 | USD +42.50  |
| production strict fill | RTH     | up        | 1    | 1    | 100.0%   | USD +800 | USD +800.00 |

| metric                  | legacy superseded output    | corrected current output    | status                       |
| ----------------------- | --------------------------- | --------------------------- | ---------------------------- |
| S2 candidates           | 267 snapshots               | 233 unique semantic keys    | current proxy only           |
| S2 failed / unavailable | 251 / 4             

### GTH confirmation → virtual-open freshness

This lifecycle integrity audit is independent of replay profitability and remains valid after the backtest correction.

In [5]:
gth_by_id = {row["event_id"]: row for row in gth_signals}
gth_opens = [
    row for row in virtual_events
    if row.get("event") == "virtual_opened" and row.get("source_kind") == "gth_dip_reclaim_call"
]
opens_by_signal = defaultdict(list)
for row in gth_opens:
    opens_by_signal[row["source_signal_id"]].append(row)
assert len(gth_by_id) == len(gth_signals) == 6
assert set(opens_by_signal) == set(gth_by_id)
assert all(len(rows) == 1 for rows in opens_by_signal.values())

lag_rows = []
for signal_id, signal_row in sorted(gth_by_id.items(), key=lambda pair: pair[1]["confirmed_at"]):
    open_row = opens_by_signal[signal_id][0]
    confirmed = parse_ts(signal_row["confirmed_at"])
    opened = parse_ts(open_row["opened_at"])
    persisted_until = parse_ts(signal_row.get("valid_until"))
    valid_until = persisted_until or confirmed + timedelta(seconds=GTH_TTL_SECONDS)
    lag = (opened - confirmed).total_seconds()
    if opened > valid_until:
        validity = "STALE"
    elif lag < -CLOCK_TOLERANCE_SECONDS:
        validity = "INVALID_CLOCK"
    elif lag < 0:
        validity = "fresh_clock_skew"
    else:
        validity = "fresh"
    lag_rows.append((signal_id, confirmed, opened, lag, validity, persisted_until))

md_table(
    ("signal_id", "confirmed_at", "opened_at", "lag_s", "validity"),
    [
        (signal_id, confirmed.isoformat(), opened.isoformat(), f"{lag:.3f}", validity)
        for signal_id, confirmed, opened, lag, validity, persisted_until in lag_rows
    ],
)
stale = [row for row in lag_rows if row[4] == "STALE"]
fresh = [row for row in lag_rows if row[4] in {"fresh", "fresh_clock_skew"}]
lag_values = [row[3] for row in lag_rows]
print()
print(f"Join coverage: {len(lag_rows)}/{len(gth_signals)}")
print(f"Within validity including tolerated clock skew: {len(fresh)}/{len(lag_rows)}")
print(f"Over valid TTL / 600s: {len(stale)}/{len(lag_rows)} ({len(stale) / len(lag_rows):.1%})")
print(f"Lag seconds min / median / max: {min(lag_values):.3f} / {median(lag_values):.3f} / {max(lag_values):.3f}")
print(f"Historical signals missing persisted valid_until: {sum(row[5] is None for row in lag_rows)}/{len(lag_rows)}")


| signal_id                        | confirmed_at                     | opened_at                        | lag_s    | validity         |
| -------------------------------- | -------------------------------- | -------------------------------- | -------- | ---------------- |
| gth-dip:eea924379316ad989bac90d0 | 2026-07-15T07:39:15.391000+00:00 | 2026-07-15T07:53:01.851555+00:00 | 826.461  | STALE            |
| gth-dip:35c08b2ada1fab21e83b258d | 2026-07-16T06:09:55.427000+00:00 | 2026-07-16T06:09:53.905751+00:00 | -1.521   | fresh_clock_skew |
| gth-dip:6f0c0a49585b1f77eafd9ab2 | 2026-07-16T08:33:22.412000+00:00 | 2026-07-16T10:01:03.872556+00:00 | 5261.461 | STALE            |
| gth-dip:be539899199cba09f802b2d2 | 2026-07-16T11:53:49.449000+00:00 | 2026-07-16T11:53:52.879264+00:00 | 3.430    | fresh            |
| gth-dip:cf1caf827b15e5bbea3e99bd | 2026-07-17T04:51:50.026000+00:00 | 2026-07-17T04:51:54.671119+00:00 | 4.645    | fresh            |
| gth-dip:809431a8ce708bbffde3a67c | 2026

### Data-quality checks and limitations

The final checks reconcile the current artifact and label what can and cannot support a production decision.

In [6]:
current_s2_total = sum(float(row["pnl_usd"]) for row in s2_rows)
current_prod_total = sum(float(row["pnl_usd"]) for row in production_rows)
unique_intent_events = {row["event_id"] for row in trade_intents}
unique_contexts = {row["context_id"] for row in trade_intents}
assert current_s2_total == s2["total_pnl_usd"] == 290.0
assert current_prod_total == production["total_pnl_usd"] == 800.0
assert all(row["_primary_at"] is None or row["_primary_at"] < CUTOFF_AT for row in trade_intents + gth_signals + virtual_events)
assert all(row["automatic_ordering"] is False for row in ready_rows)

quality_rows = (
    ("Current artifact reconciliation", "S2 7 / USD +290; production 1 / USD +800", "pass"),
    ("Production cohort isolation", "confirmed/prefill/gth_dip explicitly excluded", "pass"),
    ("trade_intent grain", f"474 rows / {len(unique_intent_events)} event IDs / {len(unique_contexts)} contexts", "not a conversion funnel"),
    ("Strict fill meaning", "quote-lake ask <= saved limit; no real order was sent", "simulated replay"),
    ("Put/down production case", "target reached before entry; no fill and no PnL", "not a losing trade"),
    ("S2 evidence role", "233 unique candidates; 7 replay rows", "observational proxy only"),
    ("GTH validity", "6/6 historical signals lack persisted valid_until", "600s reconstructed"),
    ("Holdout size", "5 complete sessions; 2 ready intents; 1 replay fill", "no edge claim"),
    ("Automatic ordering", "False for both ready decisions", "unchanged"),
    ("Look-ahead gate check", "artifact says outcome_horizons_used_as_gate=false", "pass"),
)
md_table(("check", "observed", "assessment"), quality_rows)
print()
print("Confidence: ready for engineering diagnosis; not ready for automatic execution or strategy-edge claims.")


| check                           | observed                                              | assessment               |
| ------------------------------- | ----------------------------------------------------- | ------------------------ |
| Current artifact reconciliation | S2 7 / USD +290; production 1 / USD +800              | pass                     |
| Production cohort isolation     | confirmed/prefill/gth_dip explicitly excluded         | pass                     |
| trade_intent grain              | 474 rows / 280 event IDs / 440 contexts               | not a conversion funnel  |
| Strict fill meaning             | quote-lake ask <= saved limit; no real order was sent | simulated replay         |
| Put/down production case        | target reached before entry; no fill and no PnL       | not a losing trade       |
| S2 evidence role                | 233 unique candidates; 7 replay rows                  | observational proxy only |
| GTH validity                    | 6/6 historic

## Takeaways

1. **Ship the deterministic safety fix:** fail closed on missing, expired, or materially future GTH confirmations. The stale-open evidence is 2/6.
2. **Use the production cohort correctly:** only `trade_ready × baseline × naked` is production strategy PnL. Its current result is one replayed fill (+$800) and one `target_before_entry`; this is far too small for direction or session tuning.
3. **Keep S2 in shadow:** corrected S2 is +$290 across seven rows, including down/put +$120 across three rows. The reversal from the legacy negative result demonstrates measurement sensitivity, not a newly proven edge.
4. **Retire old diagnoses:** do not reuse 267/12/-$1,560, GTH 12/-$3,250, or put/down 8/-$1,730 as current evidence.
5. **Keep `automatic_ordering=False`** until naturally persisted, walk-forward production decisions and fill/no-fill outcomes span substantially more complete sessions.